# Notebook 4 - Clean Holdings and Classify Assets

two main things happen here:

**1. deduplication**

berkshire reports the same stock in multiple rows because they have different subsidiaries managing different chunks of the same position. apple for example shows up 2-3 times per quarter. we need to add those rows together to get the actual position size. we group by cusip and put_call (keeping put_call separate because a put option on a stock is completely different from owning the stock).

**2. asset classification**

the titleOfClass field tells us what type of security it is but its messy text like "COM" or "PFD" or "CALL". we map these to clean labels like Common Stock, Preferred Stock, Call Option etc. this lets us do proper asset allocation analysis later.

after this each row is one unique position per quarter with the correct total shares and value.

In [ ]:
import os, sys, pathlib

# ----- COLAB SETUP -----
# Option A: with Google Drive (data stays between sessions)
# from google.colab import drive
# drive.mount("/content/drive")
# PROJECT_ROOT = pathlib.Path("/content/drive/MyDrive/13F-Analytics")

# Option B: without Drive (data gone when session ends, need to rerun)
# PROJECT_ROOT = pathlib.Path("/content/13F-Analytics")

# detects root automatically when running locally
PROJECT_ROOT = pathlib.Path("/content/13F-Analytics") if pathlib.Path("/content/13F-Analytics").exists() else pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()

os.chdir(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("working directory:", PROJECT_ROOT)

In [ ]:
from src.assets import clean_holdings

holdings = clean_holdings()
holdings.head(10)

In [ ]:
# positions that were merged from multiple rows (the berkshire duplicate problem)
# n_rows > 1 means that many raw rows got combined into this one position
holdings[holdings["n_rows"] > 1][["quarter", "issuer", "cusip", "n_rows", "shares", "value_usd"]].head(10)

In [ ]:
# portfolio weights should add up to 1.0 for each quarter
holdings.groupby("quarter")["portfolio_weight"].sum().round(4)

In [ ]:
# what types of securities does this manager hold
latest = holdings["quarter"].max()
holdings[holdings["quarter"] == latest].groupby("asset_class")["value_usd"].sum().sort_values(ascending=False)